This notebook can reproduce performance comparison results reported in the Appendix of the paper for ReLU networks.
Use the appropriate choice of hyperparameters to get the best accuracies.

#### Imports

In [1]:
import numpy as np
import torch
import torch.nn as nn
import os
from tqdm import tqdm
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F

#### Hyper-parameters

In [ ]:
#Hyper Parameters and Run Parameters
config = dict(
    seed = 42,
    dataset = '5a',
    input_dim = 100, #input correct dimensions for the data
    output_dim = 2,
    hidden_config = [40, 40, 40, 40, 40], 
    bias = True,
    lr = 0.005,
    epochs = 400, 
    num_batches = 1000,
    num_workers = 0, 
    decay = 0.01,
    sched = 'step',
    patience = 50,
    early_stop = 'Y',
    milestones = [300],
    opt = 'AdamW',
    init = 'D',
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    )
run_specs = dict(
    run_name = 'l5-20n', 
    group = 'ReLU',
    use_artifact = 'N',
    save_interval = [5000],
    # save_interval = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 20, 30, 40, 50, 75, 100, 150, 200, 250, 300, 301, 350, 400],
    print = 40)


#### Util Functions

In [3]:
#Seeding
def set_torchseed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)

In [4]:
#Test Function
def relutest(model, X_test, y_test, device):

    model.eval()
    loss_fn = nn.CrossEntropyLoss()
    
    # Create test dataloader
    total_loss = 0.0
    # all_predictions = []
    # all_gate_scores = []
    # all_activations = []
    correct = 0
    total = 0
    
    with torch.no_grad():
    
        batch_data, batch_labels = X_test.to(device), y_test.to(device)

        # Forward pass
        y_logits = model(batch_data)
        y_pred = torch.argmax(y_logits, dim=1)
        
        # Calculate metrics
        loss = loss_fn(y_logits, batch_labels)
        total_loss += loss.item()
        
        # Update counts
        total += batch_labels.size(0)
        correct += torch.eq(batch_labels, y_pred).sum().item()
  
    # Calculate final metrics
    accuracy = 100 * correct / total
    # No need to divide by len(X_test) because Cross Entropy return average by default.

    return total_loss, accuracy

In [5]:
#Training Function
def relutrain(model, X_train, y_train, X_test, y_test, config, run_specs, model_dir=None):

    if 0 in run_specs['save_interval']:
        if not os.path.exists(model_dir):
            os.makedirs(model_dir)
        torch.save(model.state_dict(), f'{model_dir}/model_epoch_0.pth')
        
    loss_fn = nn.CrossEntropyLoss()

    param_groups = [
    {"params": list(model.parameters()), "weight_decay": config['decay']}
    ]

    # Initialize optimizer
    if config['opt'].lower() == 'adamw':
        optimizer = torch.optim.AdamW(param_groups, lr=config['lr'])
    # Initialize optimizer
    elif config['opt'].lower() == 'sgd':
        optimizer = torch.optim.SGD(param_groups, lr=config['lr'], momentum=0.9, nesterov=True)

    # Initialize scheduler
    if config['sched'] == 'step':
        scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=config['milestones'], gamma=0.1)
    elif config['sched'] == 'cosine':
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'])
    else:
        scheduler = None

    batch_size = int(len(X_train)/ config['num_batches'])
    train_dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size,
        shuffle=True,
        pin_memory=True,
        drop_last=False,
        num_workers = config['num_workers']
    )

    patience_counter = 0
    best_val_acc = 0
    for epoch in tqdm(range(1, config['epochs']+1)):

        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        for batch_data, batch_labels in train_loader:
            
            batch_data, batch_labels = batch_data.to(config['device'], non_blocking=True), batch_labels.to(config['device'], non_blocking=True)   
             
            # Forward pass
            y_logits = model(batch_data)
        
            # Predictions
            y_pred = torch.argmax(y_logits, dim=1) 
            total += batch_labels.size(0)
            loss = loss_fn(y_logits, batch_labels)  
            running_loss += loss.item()
            correct += y_pred.eq(batch_labels).sum().item()

            # Backward pass
            optimizer.zero_grad()
            loss.backward()                        
            optimizer.step()

        if scheduler is not None:
            scheduler.step()
            
        train_accuracy = 100 * correct / total
        train_loss = running_loss/ config['num_batches']            

        log_dict = {"Train_Loss": train_loss, "Train_Accuracy": train_accuracy, "Epoch": epoch}
        
        if epoch % run_specs['print'] == 0:
            val_loss, val_acc = relutest(model, X_test, y_test, config['device'])
            if (val_acc < best_val_acc) and (epoch < 100):
                patience_counter += 1
                if patience_counter >= config['patience']:
                    print(f"Epoch: {epoch} | Train Loss: {train_loss:.5f}, Train Accuracy: {train_accuracy:.2f}%")
                    print(f"Validation Loss {val_loss} | Validation Accuracy : {val_acc}")
                    log_dict.update({"Validation_Loss": val_loss, "Validation_Accuracy": val_acc})
                    print("Early stopping triggered")           
                    return
            else:
                best_val_acc = val_acc
                patience_counter = 0

            if ((epoch >= 50 and val_acc < 60) or (epoch >= 100 and val_acc < 70) or (epoch >= 200 and val_acc < 80)) and (config['early_stop'] == 'Y'):
                print(f"Epoch: {epoch} | Train Loss: {train_loss:.5f}, Train Accuracy: {train_accuracy:.2f}%") 
                print(f"Validation Loss {val_loss} | Validation Accuracy : {val_acc}")
                log_dict.update({"Validation_Loss": val_loss, "Validation_Accuracy": val_acc})
                print('Early Stopping ..')

                return
                
            if config['sched'] == 'N':
                print(f"Epoch: {epoch} | Train Loss: {train_loss:.5f}, Train Accuracy: {train_accuracy:.2f}%")
            else:
                current_lr = scheduler.get_last_lr()[0]
                print(f"Epoch: {epoch} | Train Loss: {train_loss:.5f}, Train Accuracy: {train_accuracy:.2f}% , lr:{current_lr:.6f}")
            print(f"Validation Loss {val_loss} | Validation Accuracy : {val_acc}")
            log_dict.update({"Validation_Loss": val_loss, "Validation_Accuracy": val_acc})
            
        # Save model as artifact at save_interval
        if epoch in run_specs['save_interval']:
            if not os.path.exists(model_dir):
                os.makedirs(model_dir)
            torch.save(model.state_dict(), f'{model_dir}/model_epoch_{epoch}.pth')


#### Dataset

In [ ]:
def get_decision(dataset):

    if dataset == '5a':
        path = './data/level_5a.pth'
    else:
        raise ValueError(f"Dataset '{dataset}' does not exist.")
    
    # Load the dataset
    loaded_data = torch.load(path)

    # Access the components
    X = loaded_data["data"]
    y = loaded_data["labels"]
    tree_planes = loaded_data["tree_planes"]
    num_data = X.shape[0]
    num_data = len(X)
    num_train= num_data//2
    num_vali = num_data//4
    num_test = num_data//4
    train_data = X[:num_train,:]
    train_data_labels = y[:num_train]

    vali_data = X[num_train:num_train+num_vali,:]
    vali_data_labels = y[num_train:num_train+num_vali]

    test_data = X[num_train+num_vali :,:]
    test_data_labels = y[num_train+num_vali :]

    if (dataset == '5b') or (dataset == '5c'):
        shuffled_indices = torch.randperm(train_data.shape[0])
        train_data = train_data[shuffled_indices]
        train_data_labels = train_data_labels[shuffled_indices]

    return train_data, train_data_labels, vali_data, vali_data_labels, test_data, test_data_labels, tree_planes


In [7]:
#Dataset
train_data, train_data_labels, val_data, val_data_labels, test_data, test_data_labels, tree_planes = get_decision(config['dataset'])

#### Model

In [8]:
class MLPReLU(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_config, bias=False):
        super().__init__()

        self.num_nodes = [input_dim] + hidden_config + [output_dim]

        self.layers = nn.ModuleList()

        for i in range(len(hidden_config)+1):
            self.layers.append(nn.Linear(self.num_nodes[i], self.num_nodes[i+1], bias=bias))
    
    def forward(self, x):

        for i in range(len(self.layers)-1):
            x = self.layers[i](x)
            x = F.relu(x)
        
        x = self.layers[-1](x)

        return x


In [9]:
#Model Declaration
model = MLPReLU(input_dim = config['input_dim'],
    output_dim = config['output_dim'],
    hidden_config = config['hidden_config'],
    bias=config['bias']).to(config['device'])

#### Training

In [10]:
set_torchseed(config['seed'])
# save_dir = './models' #Enter path of directory to save the models
save_dir = None

In [11]:
relutrain(model, train_data, train_data_labels, val_data, val_data_labels, config, run_specs, save_dir)

 10%|█         | 40/400 [01:50<16:36,  2.77s/it]

Epoch: 40 | Train Loss: 0.54555, Train Accuracy: 73.26% , lr:0.005000
Validation Loss 0.49516546726226807 | Validation Accuracy : 76.33666666666667


 20%|██        | 80/400 [03:40<14:50,  2.78s/it]

Epoch: 80 | Train Loss: 0.50118, Train Accuracy: 77.11% , lr:0.005000
Validation Loss 0.4146425127983093 | Validation Accuracy : 82.21833333333333


 30%|███       | 120/400 [05:29<12:36,  2.70s/it]

...

Epoch: 320 | Train Loss: 0.35092, Train Accuracy: 87.79% , lr:0.000500
Validation Loss 0.22684219479560852 | Validation Accuracy : 92.86333333333333


 90%|█████████ | 360/400 [16:28<01:47,  2.69s/it]

Epoch: 360 | Train Loss: 0.34431, Train Accuracy: 88.21% , lr:0.000500
Validation Loss 0.22444568574428558 | Validation Accuracy : 93.1


100%|██████████| 400/400 [18:18<00:00,  2.75s/it]

Epoch: 400 | Train Loss: 0.34569, Train Accuracy: 88.17% , lr:0.000500
Validation Loss 0.22472000122070312 | Validation Accuracy : 93.15833333333333
